# Agent Skills 与 Agent-facing CLI：两个独立能力及其组合方式

> 本章先独立讲 Agent-facing CLI，再独立讲 Agent Skills。CLI 案例把日历 REST 服务改造成稳定执行接口；Skill 案例让真实模型完成发现、触发、加载、脚本执行和结果生成。


## 1 结论与学习目标

本章包含两个知识点：

1. **Agent-facing CLI**：面向 Agent 的确定性执行接口。重点是命令、Schema、认证授权、结构化输出、退出码、分页、dry-run、幂等和审计。
2. **Agent Skill**：面向 Agent 的可复用能力包。重点是发现、触发、渐进加载、指令、脚本、参考资料、资产、执行、评测和分发。

它们可以独立使用：

- CLI 可以直接被 Agent、普通程序、CI 或 MCP Server 调用。
- Skill 可以调用文件工具、浏览器、MCP、现有软件包、内置脚本或 CLI。
- Skill 与 CLI 搭配时，Skill 提供流程知识，CLI 提供稳定业务动作。

完成本章后，你应该能够：

1. 把 Web/API 能力设计成 Agent-facing CLI。
2. 解释 Skill 从编写到下线的完整生命周期。
3. 判断何时使用 `SKILL.md`、`scripts/`、`references/`、`assets/` 和 `agents/openai.yaml`。
4. 让模型根据 Skill 元数据选择能力，按需加载正文和资源，并消费脚本 Observation。
5. 独立评测触发准确率、任务质量、脚本可靠性、Token 和延迟。


### 1.1 本章边界

本章不讲传统 Linux 命令基础，也不把 Skill 简化成 Prompt 模板。

CLI 实验完全离线。Skill 实验使用“需求文档质量审查”案例，核心生命周期接入 DeepSeek/OpenAI-compatible API；`RUN_LIVE_LLM=0` 时跳过真实请求，便于仓库验证。

完整 Harness、沙箱和审批系统留到第 10 章。本章只实现理解 Skill 生命周期所需的最小 Host。


## 2 两个知识点的边界

### 2.1 Skill 与 CLI 分别解决什么

| 维度 | Agent Skill | Agent-facing CLI |
| --- | --- | --- |
| 核心产物 | 指令和按需资源目录 | 稳定命令接口 |
| 主要消费者 | Agent 模型与 Host | Agent、程序、CI、MCP Server、人 |
| 解决问题 | Agent 何时做、按什么步骤做、失败后看什么 | 程序怎样确定地执行动作并返回结果 |
| 生命周期 | 编写、发现、触发、加载、执行、评测、分发、更新 | 设计、发布、认证、调用、兼容、监控、下线 |
| 是否必须调用模型 | 运行时通常由模型解释和选择 | 不需要 |
| 是否必须提供进程命令 | 不需要 | 需要稳定的进程或等价命令边界 |
| 权限边界 | 不能单独充当权限边界 | 应强制认证、授权和审计 |

Skill 内的 `scripts/` 与 Agent-facing CLI 也要区分：

- Skill Script 是能力包内部的实现资源，服务于该 Skill，可以随 Skill 一起修改。
- Agent-facing CLI 是独立产品接口，需要稳定版本、认证授权、兼容策略和跨调用方支持。


### 2.2 独立使用与组合使用

```mermaid
flowchart LR
    A["Agent"] --> S["Skill：流程知识"]
    A --> C["Agent-facing CLI：执行接口"]
    S --> T["文件 / 浏览器 / MCP / Script"]
    S -. "可选组合" .-> C
    C --> P["业务 API"]
```

学习顺序先分后合：第 3 至 6 节只讲 CLI，第 7 至 9 节只讲 Skill，第 10 节再讨论组合。


## 3 Agent-facing CLI：背景与契约

网页适合人操作，REST API 适合程序集成。Agent 直接操作网页会受到页面结构、弹窗和加载状态影响；直接调用 REST API 又会在每个 Agent 客户端重复实现认证、分页、错误映射和版本适配。

Agent-facing CLI 在业务 API 上提供稳定适配：

- 参数由 JSON 和 Schema 描述。
- stdout 返回机器可读结果。
- stderr 返回诊断信息。
- 退出码和错误码区分恢复策略。
- 写操作提供 dry-run、幂等和审计。

CLI 的价值来自接口契约，与终端界面无关。桌面 Agent、IDE、CI 和 MCP Server 都可以调用同一个命令面。


### 3.1 最小生产契约

| 契约 | 推荐设计 | 原因 |
| --- | --- | --- |
| 命令 | `资源.动作` | 名称稳定，便于发现和生成 |
| 查询输入 | `--params` JSON | 保留类型，减少字符串解析 |
| 请求体 | `--json` JSON | 与 REST 请求体对应 |
| 成功输出 | stdout JSON/NDJSON | Agent 可稳定解析 |
| 诊断 | stderr | 不污染业务结果 |
| 状态 | 退出码 + `error.code` | Harness 可选择修参、登录、重试或停止 |
| 探索 | `schema <command>` | 调用前获取参数契约 |
| 大结果 | 分页令牌或 NDJSON | 控制内存和上下文 |
| 写操作 | dry-run + 幂等键 | 先预演，重试不重复写入 |
| 版本 | `api_version` | 支持兼容演进 |


### 3.2 参与者与数据流

```mermaid
sequenceDiagram
    participant A as Agent
    participant H as Harness
    participant C as CLI
    participant P as Business API
    A->>H: 结构化动作意图
    H->>C: argv + account + timeout
    C->>C: 参数、身份和权限校验
    C->>P: REST / RPC 请求
    P-->>C: 业务结果或错误
    C-->>H: exit code + stdout + stderr
    H-->>A: 结构化 Observation
```

模型提出动作，Harness 管理进程，CLI 强制执行契约，业务 API 维护最终业务规则。


## 4 CLI 的认证与授权

认证回答“调用者是谁”，授权回答“调用者能做什么”。多用户 Agent 还要绑定用户、租户、账户、会话和审计记录。

本地桌面 Agent 常用 OAuth 授权码流程：浏览器完成登录和同意，CLI 保存短期访问令牌与刷新凭据。凭据放入操作系统钥匙串、密钥管理服务或工作负载身份系统，不能进入 Skill、命令参数、Prompt、日志和 Checkpoint。[6]


### 4.1 身份方案与取舍

| 场景 | 建议身份 | 主要取舍 |
| --- | --- | --- |
| 单用户本地 Agent | 用户 OAuth + 系统钥匙串 | 权限与用户一致；需要登录和刷新 |
| 多用户 SaaS Agent | 每用户独立 OAuth 凭据 | 隔离明确；生命周期管理复杂 |
| 企业后台任务 | 工作负载身份或服务账户 | 无人值守；必须限制资源和 Scope |
| CI 临时任务 | 短期联合身份 | 减少长期密钥；基础设施配置更多 |

执行原则：最小 Scope、显式账户、租户隔离、短期凭据、写操作审计、撤销和过期错误分离。


## 5 CLI 实验：把团队日历 REST 服务改造成 `workcli`

团队日历已有以下接口：

```http
GET  /v1/events?page_size=20&page_token=...
POST /v1/events
```

我们保留业务 API，在前面增加命令面：

```text
workcli schema calendar.events.create
workcli calendar.events.list --params '{"page_size": 2}' --format json
workcli calendar.events.create --json '{...}' --dry-run
```

验收条件：读写 Scope 分离；所有结果可解析；错误可分类；写操作支持预演和幂等重试。


### 5.1 准备数据结构

以下代码全部离线运行。`CalendarService` 代替真实 REST 服务，Notebook 直接传入参数列表，避免 shell 引号干扰观察。


In [1]:
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any


API_VERSION = "v1"


@dataclass(frozen=True)
class AuthContext:
    """CLI 从安全存储读取后的身份上下文；这里不保存真实令牌。"""

    account: str
    authenticated: bool
    scopes: frozenset[str]


@dataclass
class CliResult:
    """模拟进程的三个可观察结果：退出码、stdout 和 stderr。"""

    exit_code: int
    stdout: str = ""
    stderr: str = ""

    def json(self) -> dict[str, Any]:
        return json.loads(self.stdout) if self.stdout else {}


class CliError(Exception):
    def __init__(self, exit_code: int, code: str, message: str, details: dict | None = None):
        super().__init__(message)
        self.exit_code = exit_code
        self.code = code
        self.message = message
        self.details = details or {}


### 5.2 错误契约与业务服务

退出码供 Harness 快速分支，`error.code` 供 Agent 选择恢复动作：2 表示修正参数，3 表示登录，4 表示补权限，5 表示处理业务冲突，6 表示按策略重试上游故障。


In [2]:
@dataclass
class CalendarService:
    """内存版业务服务；生产实现会把这些方法替换为 REST 请求。"""

    events: list[dict[str, Any]] = field(default_factory=list)
    idempotency_records: dict[str, dict[str, Any]] = field(default_factory=dict)

    def list_events(self, page_size: int, page_token: str | None) -> tuple[list[dict], str | None]:
        start = int(page_token or 0)
        end = start + page_size
        next_token = str(end) if end < len(self.events) else None
        return self.events[start:end], next_token

    def validate_create(self, body: dict[str, Any]) -> None:
        required = {"title", "start", "end", "attendees"}
        missing = sorted(required - body.keys())
        if missing:
            raise CliError(2, "INVALID_ARGUMENT", "缺少必填字段", {"missing": missing})

        unknown = sorted(body.keys() - required)
        if unknown:
            raise CliError(2, "INVALID_ARGUMENT", "存在未知字段", {"unknown": unknown})
        if not isinstance(body["title"], str) or not body["title"].strip():
            raise CliError(2, "INVALID_ARGUMENT", "title 必须是非空字符串")

        try:
            start = datetime.fromisoformat(body["start"])
            end = datetime.fromisoformat(body["end"])
        except (TypeError, ValueError) as exc:
            raise CliError(2, "INVALID_DATETIME", "start 和 end 必须是 ISO 8601 时间") from exc

        if start.tzinfo is None or end.tzinfo is None:
            raise CliError(2, "TIMEZONE_REQUIRED", "start 和 end 必须包含时区")
        if start >= end:
            raise CliError(2, "INVALID_TIME_RANGE", "end 必须晚于 start")
        if not isinstance(body["attendees"], list) or not all(
            isinstance(attendee, str) and attendee for attendee in body["attendees"]
        ):
            raise CliError(2, "INVALID_ARGUMENT", "attendees 必须是字符串数组")

    def find_conflicts(self, body: dict[str, Any]) -> list[dict[str, Any]]:
        start = datetime.fromisoformat(body["start"])
        end = datetime.fromisoformat(body["end"])
        conflicts = []
        for event in self.events:
            old_start = datetime.fromisoformat(event["start"])
            old_end = datetime.fromisoformat(event["end"])
            if start < old_end and end > old_start:
                conflicts.append(event)
        return conflicts

    def create_event(self, body: dict[str, Any], idempotency_key: str) -> tuple[dict, bool]:
        # 同一个幂等键再次提交时返回第一次的结果，避免超时重试产生重复会议。
        if idempotency_key in self.idempotency_records:
            record = self.idempotency_records[idempotency_key]
            if record["body"] != body:
                raise CliError(5, "IDEMPOTENCY_KEY_REUSED", "同一幂等键不能对应不同请求体")
            return record["event"], True

        self.validate_create(body)
        conflicts = self.find_conflicts(body)
        if conflicts:
            raise CliError(
                5,
                "EVENT_CONFLICT",
                "目标时间已有日程",
                {"event_ids": [event["id"] for event in conflicts]},
            )

        event = {"id": f"evt-{len(self.events) + 1:03d}", **body}
        self.events.append(event)
        self.idempotency_records[idempotency_key] = {"body": body.copy(), "event": event}
        return event, False


service = CalendarService(
    events=[
        {
            "id": "evt-001",
            "title": "需求评审",
            "start": "2026-07-20T09:00:00+08:00",
            "end": "2026-07-20T10:00:00+08:00",
            "attendees": ["pm@example.com", "rd@example.com"],
        },
        {
            "id": "evt-002",
            "title": "迭代复盘",
            "start": "2026-07-20T14:00:00+08:00",
            "end": "2026-07-20T15:00:00+08:00",
            "attendees": ["team@example.com"],
        },
        {
            "id": "evt-003",
            "title": "技术方案讨论",
            "start": "2026-07-21T10:00:00+08:00",
            "end": "2026-07-21T11:00:00+08:00",
            "attendees": ["rd@example.com", "qa@example.com"],
        },
    ]
)


### 5.3 Schema 与参数解析

Agent 调用前先读取 Schema。真实 CLI 可以从 OpenAPI、Discovery Document 或手写 Schema 生成命令契约。


In [3]:
COMMAND_SCHEMAS: dict[str, dict[str, Any]] = {
    "calendar.events.list": {
        "description": "分页读取事件",
        "params": {
            "type": "object",
            "properties": {
                "page_size": {"type": "integer", "minimum": 1, "maximum": 100, "default": 20},
                "page_token": {"type": ["string", "null"]},
            },
            "additionalProperties": False,
        },
        "required_scope": "calendar.events.read",
    },
    "calendar.events.create": {
        "description": "创建事件",
        "body": {
            "type": "object",
            "required": ["title", "start", "end", "attendees"],
            "properties": {
                "title": {"type": "string", "minLength": 1},
                "start": {"type": "string", "format": "date-time"},
                "end": {"type": "string", "format": "date-time"},
                "attendees": {"type": "array", "items": {"type": "string"}},
            },
            "additionalProperties": False,
        },
        "required_scope": "calendar.events.write",
        "requires_idempotency_key": True,
        "supports_dry_run": True,
    },
}


class SafeArgumentParser(argparse.ArgumentParser):
    """argparse 默认会退出 Notebook；改为抛出可映射的业务异常。"""

    def error(self, message: str) -> None:
        raise CliError(2, "CLI_USAGE_ERROR", message)


def build_parser() -> argparse.ArgumentParser:
    parser = SafeArgumentParser(prog="workcli", add_help=False)
    parser.add_argument("method", help="schema 或资源.动作")
    parser.add_argument("target", nargs="?", help="schema 要查看的命令")
    parser.add_argument("--params", default="{}", help="查询参数 JSON")
    parser.add_argument("--json", dest="json_body", default="{}", help="请求体 JSON")
    parser.add_argument("--format", choices=["json", "ndjson"], default="json")
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--idempotency-key")
    return parser


### 5.4 命令分发器

分发器完成 JSON 解析、认证、Scope 校验、服务调用和结果格式化。模型输出不能越过这些校验直接进入业务 API。


In [4]:
def parse_json(raw: str, argument_name: str) -> dict[str, Any]:
    try:
        value = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise CliError(2, "INVALID_JSON", f"{argument_name} 不是合法 JSON", {"position": exc.pos}) from exc
    if not isinstance(value, dict):
        raise CliError(2, "INVALID_ARGUMENT", f"{argument_name} 必须是 JSON 对象")
    return value


def require_scope(auth: AuthContext, scope: str) -> None:
    if not auth.authenticated:
        raise CliError(3, "AUTH_REQUIRED", "当前账户未登录")
    if scope not in auth.scopes:
        raise CliError(4, "SCOPE_MISSING", f"缺少权限：{scope}", {"required_scope": scope})


def success_payload(data: Any, request_id: str = "req-demo") -> dict[str, Any]:
    return {"api_version": API_VERSION, "request_id": request_id, "data": data}


def format_error(error: CliError) -> CliResult:
    payload = {
        "api_version": API_VERSION,
        "error": {"code": error.code, "message": error.message, "details": error.details},
    }
    return CliResult(
        exit_code=error.exit_code,
        stdout=json.dumps(payload, ensure_ascii=False),
        stderr=f"workcli: {error.code}: {error.message}",
    )


def run_cli(argv: list[str], auth: AuthContext, calendar: CalendarService) -> CliResult:
    try:
        args = build_parser().parse_args(argv)

        if args.method == "schema":
            if args.target not in COMMAND_SCHEMAS:
                raise CliError(2, "UNKNOWN_COMMAND", f"未知命令：{args.target}")
            payload = success_payload(COMMAND_SCHEMAS[args.target])
            return CliResult(0, json.dumps(payload, ensure_ascii=False, indent=2))

        if args.method not in COMMAND_SCHEMAS:
            raise CliError(2, "UNKNOWN_COMMAND", f"未知命令：{args.method}")

        schema = COMMAND_SCHEMAS[args.method]
        require_scope(auth, schema["required_scope"])

        if args.method == "calendar.events.list":
            params = parse_json(args.params, "--params")
            unknown = sorted(set(params) - {"page_size", "page_token"})
            if unknown:
                raise CliError(2, "INVALID_ARGUMENT", "存在未知查询参数", {"unknown": unknown})
            page_size = params.get("page_size", 20)
            if isinstance(page_size, bool) or not isinstance(page_size, int) or not 1 <= page_size <= 100:
                raise CliError(2, "INVALID_ARGUMENT", "page_size 必须在 1 到 100 之间")
            page_token = params.get("page_token")
            if page_token is not None:
                if not isinstance(page_token, str) or not page_token.isdigit():
                    raise CliError(2, "INVALID_PAGE_TOKEN", "page_token 无效")
            items, next_token = calendar.list_events(page_size, page_token)

            if args.format == "ndjson":
                # NDJSON 每行都是独立 JSON；最后一行携带分页信息，适合流式消费。
                records = [{"type": "event", "data": item} for item in items]
                records.append({"type": "page", "next_page_token": next_token})
                stdout = "\n".join(json.dumps(row, ensure_ascii=False) for row in records)
                return CliResult(0, stdout)

            data = {"items": items, "next_page_token": next_token}
            return CliResult(0, json.dumps(success_payload(data), ensure_ascii=False, indent=2))

        body = parse_json(args.json_body, "--json")
        calendar.validate_create(body)
        conflicts = calendar.find_conflicts(body)

        if args.dry_run:
            data = {
                "valid": not conflicts,
                "operation": "calendar.events.create",
                "normalized_body": body,
                "conflict_event_ids": [event["id"] for event in conflicts],
                "committed": False,
            }
            return CliResult(0, json.dumps(success_payload(data), ensure_ascii=False, indent=2))

        if not args.idempotency_key:
            raise CliError(2, "IDEMPOTENCY_KEY_REQUIRED", "写操作必须提供 --idempotency-key")
        event, replayed = calendar.create_event(body, args.idempotency_key)
        data = {"event": event, "replayed": replayed, "committed": True}
        return CliResult(0, json.dumps(success_payload(data), ensure_ascii=False, indent=2))

    except CliError as error:
        return format_error(error)


In [5]:
# Agent 只选择逻辑账户名；真实令牌应由 CLI 从钥匙串或密钥服务读取。
AUTH_PROFILES = {
    "product-team": AuthContext(
        account="product-team",
        authenticated=True,
        scopes=frozenset({"calendar.events.read", "calendar.events.write"}),
    ),
    "auditor": AuthContext(
        account="auditor",
        authenticated=True,
        scopes=frozenset({"calendar.events.read"}),
    ),
    "signed-out": AuthContext(
        account="signed-out",
        authenticated=False,
        scopes=frozenset(),
    ),
}


def show_result(result: CliResult) -> None:
    print(f"exit_code={result.exit_code}")
    if result.stdout:
        print("stdout:")
        print(result.stdout)
    if result.stderr:
        print("stderr:")
        print(result.stderr)


### 5.5 Schema 自省

Schema 查询不接触业务数据，因此不要求日历 Scope。


In [6]:
schema_result = run_cli(
    ["schema", "calendar.events.create"],
    AUTH_PROFILES["signed-out"],
    service,
)
show_result(schema_result)

assert schema_result.exit_code == 0
assert schema_result.json()["data"]["requires_idempotency_key"] is True


exit_code=0
stdout:
{
  "api_version": "v1",
  "request_id": "req-demo",
  "data": {
    "description": "创建事件",
    "body": {
      "type": "object",
      "required": [
        "title",
        "start",
        "end",
        "attendees"
      ],
      "properties": {
        "title": {
          "type": "string",
          "minLength": 1
        },
        "start": {
          "type": "string",
          "format": "date-time"
        },
        "end": {
          "type": "string",
          "format": "date-time"
        },
        "attendees": {
          "type": "array",
          "items": {
            "type": "string"
          }
        }
      },
      "additionalProperties": false
    },
    "required_scope": "calendar.events.write",
    "requires_idempotency_key": true,
    "supports_dry_run": true
  }
}


### 5.6 分页与 NDJSON

JSON 适合小结果；NDJSON 每行一个对象，适合大列表和流式处理。大结果应落盘，模型上下文只保存摘要和引用位置。


In [7]:
first_page = run_cli(
    ["calendar.events.list", "--params", '{"page_size": 2}', "--format", "json"],
    AUTH_PROFILES["auditor"],
    service,
)
show_result(first_page)

page_data = first_page.json()["data"]
assert first_page.exit_code == 0
assert len(page_data["items"]) == 2
assert page_data["next_page_token"] == "2"


exit_code=0
stdout:
{
  "api_version": "v1",
  "request_id": "req-demo",
  "data": {
    "items": [
      {
        "id": "evt-001",
        "title": "需求评审",
        "start": "2026-07-20T09:00:00+08:00",
        "end": "2026-07-20T10:00:00+08:00",
        "attendees": [
          "pm@example.com",
          "rd@example.com"
        ]
      },
      {
        "id": "evt-002",
        "title": "迭代复盘",
        "start": "2026-07-20T14:00:00+08:00",
        "end": "2026-07-20T15:00:00+08:00",
        "attendees": [
          "team@example.com"
        ]
      }
    ],
    "next_page_token": "2"
  }
}


In [8]:
second_page = run_cli(
    [
        "calendar.events.list",
        "--params",
        json.dumps({"page_size": 2, "page_token": page_data["next_page_token"]}),
        "--format",
        "ndjson",
    ],
    AUTH_PROFILES["auditor"],
    service,
)
show_result(second_page)

# 逐行解析，避免把完整结果作为一个巨大 JSON 一次载入。
ndjson_rows = [json.loads(line) for line in second_page.stdout.splitlines()]
assert ndjson_rows[0]["data"]["id"] == "evt-003"
assert ndjson_rows[-1]["next_page_token"] is None


exit_code=0
stdout:
{"type": "event", "data": {"id": "evt-003", "title": "技术方案讨论", "start": "2026-07-21T10:00:00+08:00", "end": "2026-07-21T11:00:00+08:00", "attendees": ["rd@example.com", "qa@example.com"]}}
{"type": "page", "next_page_token": null}


### 5.7 认证失败与授权失败

未登录应启动 OAuth；权限不足应申请增量 Scope、切换账户或拒绝任务。原样重试不会改变结果。


In [9]:
not_logged_in = run_cli(
    ["calendar.events.list"],
    AUTH_PROFILES["signed-out"],
    service,
)

write_body = {
    "title": "发布评审",
    "start": "2026-07-22T10:00:00+08:00",
    "end": "2026-07-22T10:30:00+08:00",
    "attendees": ["pm@example.com", "rd@example.com", "qa@example.com"],
}
missing_scope = run_cli(
    ["calendar.events.create", "--json", json.dumps(write_body), "--dry-run"],
    AUTH_PROFILES["auditor"],
    service,
)

print("未登录：")
show_result(not_logged_in)
print("\n权限不足：")
show_result(missing_scope)

assert not_logged_in.exit_code == 3
assert not_logged_in.json()["error"]["code"] == "AUTH_REQUIRED"
assert missing_scope.exit_code == 4
assert missing_scope.json()["error"]["code"] == "SCOPE_MISSING"


未登录：
exit_code=3
stdout:
{"api_version": "v1", "error": {"code": "AUTH_REQUIRED", "message": "当前账户未登录", "details": {}}}
stderr:
workcli: AUTH_REQUIRED: 当前账户未登录

权限不足：
exit_code=4
stdout:
{"api_version": "v1", "error": {"code": "SCOPE_MISSING", "message": "缺少权限：calendar.events.write", "details": {"required_scope": "calendar.events.write"}}}
stderr:
workcli: SCOPE_MISSING: 缺少权限：calendar.events.write


### 5.8 dry-run、幂等和业务冲突

dry-run 执行权限、Schema 和业务校验，但不提交。幂等键保证同一业务意图在超时重试后仍只产生一次写入。时间冲突返回业务错误，由 Agent 重新规划或询问用户。


In [10]:
event_count_before = len(service.events)
preview = run_cli(
    ["calendar.events.create", "--json", json.dumps(write_body), "--dry-run"],
    AUTH_PROFILES["product-team"],
    service,
)
show_result(preview)

assert preview.exit_code == 0
assert preview.json()["data"]["valid"] is True
assert preview.json()["data"]["committed"] is False
assert len(service.events) == event_count_before


exit_code=0
stdout:
{
  "api_version": "v1",
  "request_id": "req-demo",
  "data": {
    "valid": true,
    "operation": "calendar.events.create",
    "normalized_body": {
      "title": "发布评审",
      "start": "2026-07-22T10:00:00+08:00",
      "end": "2026-07-22T10:30:00+08:00",
      "attendees": [
        "pm@example.com",
        "rd@example.com",
        "qa@example.com"
      ]
    },
    "conflict_event_ids": [],
    "committed": false
  }
}


In [11]:
create_args = [
    "calendar.events.create",
    "--json",
    json.dumps(write_body),
    "--idempotency-key",
    "task-release-review-20260722",
]

first_create = run_cli(create_args, AUTH_PROFILES["product-team"], service)
retry_create = run_cli(create_args, AUTH_PROFILES["product-team"], service)

print("第一次提交：")
show_result(first_create)
print("\n同键重试：")
show_result(retry_create)

assert first_create.exit_code == 0
assert first_create.json()["data"]["replayed"] is False
assert retry_create.json()["data"]["replayed"] is True
assert first_create.json()["data"]["event"]["id"] == retry_create.json()["data"]["event"]["id"]
assert len(service.events) == event_count_before + 1

# 同一个幂等键只能代表同一个业务意图；请求体变化时必须拒绝。
changed_body = {**write_body, "title": "另一个会议"}
reused_key = run_cli(
    [
        "calendar.events.create",
        "--json",
        json.dumps(changed_body),
        "--idempotency-key",
        "task-release-review-20260722",
    ],
    AUTH_PROFILES["product-team"],
    service,
)
print("\n同键不同请求体：")
show_result(reused_key)
assert reused_key.exit_code == 5
assert reused_key.json()["error"]["code"] == "IDEMPOTENCY_KEY_REUSED"


第一次提交：
exit_code=0
stdout:
{
  "api_version": "v1",
  "request_id": "req-demo",
  "data": {
    "event": {
      "id": "evt-004",
      "title": "发布评审",
      "start": "2026-07-22T10:00:00+08:00",
      "end": "2026-07-22T10:30:00+08:00",
      "attendees": [
        "pm@example.com",
        "rd@example.com",
        "qa@example.com"
      ]
    },
    "replayed": false,
    "committed": true
  }
}

同键重试：
exit_code=0
stdout:
{
  "api_version": "v1",
  "request_id": "req-demo",
  "data": {
    "event": {
      "id": "evt-004",
      "title": "发布评审",
      "start": "2026-07-22T10:00:00+08:00",
      "end": "2026-07-22T10:30:00+08:00",
      "attendees": [
        "pm@example.com",
        "rd@example.com",
        "qa@example.com"
      ]
    },
    "replayed": true,
    "committed": true
  }
}

同键不同请求体：
exit_code=5
stdout:
{"api_version": "v1", "error": {"code": "IDEMPOTENCY_KEY_REUSED", "message": "同一幂等键不能对应不同请求体", "details": {}}}
stderr:
workcli: IDEMPOTENCY_KEY_REUSED: 同一幂等键不能对应不同请求

In [12]:
conflicting_body = {
    "title": "临时讨论",
    "start": "2026-07-20T09:30:00+08:00",
    "end": "2026-07-20T10:30:00+08:00",
    "attendees": ["pm@example.com"],
}
conflict = run_cli(
    [
        "calendar.events.create",
        "--json",
        json.dumps(conflicting_body),
        "--idempotency-key",
        "task-conflict-demo",
    ],
    AUTH_PROFILES["product-team"],
    service,
)
show_result(conflict)

assert conflict.exit_code == 5
assert conflict.json()["error"]["code"] == "EVENT_CONFLICT"
assert conflict.json()["error"]["details"]["event_ids"] == ["evt-001"]


exit_code=5
stdout:
{"api_version": "v1", "error": {"code": "EVENT_CONFLICT", "message": "目标时间已有日程", "details": {"event_ids": ["evt-001"]}}}
stderr:
workcli: EVENT_CONFLICT: 目标时间已有日程


## 6 CLI 的生产化与行业实现

### 6.1 Google Workspace CLI 对照

Google Workspace CLI（`gws`）根据 Google Discovery Service 动态构建命令面，把 Gmail、Drive、Calendar、Sheets 等 API 映射为统一 CLI，并提供 JSON、Schema、自助分页、dry-run、OAuth 和配套 Agent Skills。[5][7]

```bash
gws auth setup
gws auth login
gws schema drive.files.list
gws drive files list --params '{"pageSize": 5}'
```

截至 2026-07-17，该项目仍标注为积极开发、可能出现破坏性变更，并声明它不是受官方支持的 Google 产品。生产使用需要锁定版本并执行契约回归。[5]


### 6.2 Web/API 到 CLI 的改造步骤

1. **能力盘点**：列出资源、动作、读写属性、Scope、数据量和副作用。
2. **命令建模**：使用稳定的资源与动作，不复制网页按钮名称。
3. **数据契约**：定义 Schema、成功结构、错误结构、分页和输出上限。
4. **身份安全**：定义账户选择、凭据来源、租户隔离、审批和审计。
5. **可靠写入**：提供 dry-run、幂等、冲突检测和必要的补偿动作。
6. **可观测与版本**：记录请求 ID、版本、耗时、退出码和脱敏后的错误。
7. **契约评测**：覆盖正常、非法参数、过期令牌、权限不足、分页、冲突和超时。


### 6.3 CLI 常见踩坑

| 问题 | 后果 | 修复 |
| --- | --- | --- |
| stdout 混入日志 | JSON 随机解析失败 | 业务结果写 stdout，诊断写 stderr |
| Token 放入参数 | 命令历史和 Trace 泄露 | 只传账户别名，CLI 读取安全存储 |
| 所有失败返回 1 | Agent 无法选择恢复动作 | 退出码大类 + 稳定错误码 |
| 自动读取全部分页 | Token、延迟和费用失控 | 默认分页，显式全量 |
| 写操作缺幂等 | 超时重试产生重复数据 | 业务请求 ID 作为幂等键 |
| 隐式默认账户 | 写入错误租户 | 写操作显式选择并回显账户 |


## 7 Agent Skills：定义、组成与完整生命周期

Skill 是一个自包含目录，用于向 Agent 提供特定任务的程序性知识、领域资料和可执行资源。最小 Skill 只有 `SKILL.md`；复杂 Skill 可以增加脚本、参考资料、资产、界面元数据和评测用例。[1][2]

Prompt 只约束当前请求。Skill 可以被发现、按需加载、复用、测试、分发和更新。


### 7.1 目录结构与每种文件的职责

```text
review-requirements/
├── SKILL.md                         # 必需：触发描述和主工作流
├── agents/
│   └── openai.yaml                  # Codex 可选：UI、隐式触发策略、工具依赖
├── scripts/
│   └── extract_sections.py          # 可选：确定性、重复、易出错的操作
├── references/
│   └── review-rules.md              # 可选：按需读取的领域规则
├── assets/
│   └── review-report-template.md    # 可选：生成产物使用的模板
└── evals/
    └── evals.json                   # 建议：真实提示词、期望结果和断言
```

| 位置 | 何时加载 | 应保存什么 |
| --- | --- | --- |
| Frontmatter | Skill 发现阶段 | `name`、`description`；描述同时写“做什么”和“何时使用” |
| `SKILL.md` 正文 | Skill 触发后 | 主流程、判断条件、输入输出和失败处理 |
| `scripts/` | 需要执行时 | 可测试的确定性操作，不要求先把源码塞进上下文 |
| `references/` | 需要领域细节时 | 规则、Schema、API 文档、术语和政策 |
| `assets/` | 生成产物时 | 模板、图片、字体、样板工程和静态数据 |
| `agents/openai.yaml` | Codex 产品层 | 展示信息、隐式触发策略和工具依赖 [1] |

Agent Skills 标准允许若干可选 Frontmatter 字段；跨客户端使用时需要检查兼容性。`allowed-tools` 目前属于实验字段，不能替代运行时权限控制。[2]


### 7.2 什么时候应该写 Script

以下任务适合放入 `scripts/`：

1. 模型每次都会重复生成相似代码。
2. 输入输出可以结构化，正确性可以通过断言验证。
3. 操作容易因参数、转义、格式或顺序出错。
4. 需要固定依赖版本或稳定性能。

脚本应支持非交互执行、`--help`、结构化 stdout、明确 stderr、非零失败码、超时和边界校验。简单的一次性命令可以直接写入 `SKILL.md`；复杂命令应封装为经过测试的脚本。[3]

脚本不会自动获得权限。Host 仍要检查路径、参数、网络、文件系统范围、秘密读取和副作用。


### 7.3 从编写到下线的生命周期

```mermaid
flowchart LR
    A["1 定义任务与触发边界"] --> B["2 编写 Skill 与资源"]
    B --> C["3 静态校验和脚本测试"]
    C --> D["4 安装到 Skill Scope"]
    D --> E["5 发现元数据"]
    E --> F["6 显式或隐式触发"]
    F --> G["7 加载 SKILL.md"]
    G --> H["8 按需读取资源并执行"]
    H --> I["9 Observation 回传和完成"]
    I --> J["10 Evals 与人工复查"]
    J --> K["11 迭代、版本、分发"]
    K --> L["12 禁用、迁移或下线"]
    J -->|"未达标"| B
```

“成功运行一次”只完成了第 9 步。触发误判、脚本失败、版本漂移和质量回归通常发生在生命周期的其他阶段。


### 7.4 一次真实运行的时序

```mermaid
sequenceDiagram
    participant U as User
    participant H as Skill Host
    participant M as Model
    participant S as Skill Files
    participant P as Script
    U->>H: 审查 requirements.md
    H->>M: 用户请求 + 所有 Skill 元数据
    M-->>H: 选择 review-requirements
    H->>S: 加载 SKILL.md
    H->>M: Skill 正文 + 可用资源清单
    M-->>H: 读取规则、运行脚本、使用模板
    H->>S: 按需读取 reference 和 asset
    H->>P: 执行 extract_sections.py
    P-->>H: 结构化 Observation
    H->>M: 文档 + 规则 + Observation + 模板
    M-->>H: 审查报告
    H-->>U: 结果 + 产物
```

模型参与三个决策点：隐式触发、资源选择、最终综合。Host 负责文件读取、脚本执行、权限和结果验证。


### 7.5 渐进披露

Skill 分三层加载：[1][2]

1. **发现层**：启动时只提供名称、描述和路径。
2. **指令层**：任务匹配后加载完整 `SKILL.md`。
3. **资源层**：模型根据任务按需读取 reference、执行 script 或使用 asset。

Codex 会限制初始 Skill 列表占用的上下文；Skill 数量过多时可能缩短描述或省略部分条目。[1] 因此描述要前置关键词，Skill 正文要短，详细资料放入一层引用文件。


### 7.6 显式触发与隐式触发

| 方式 | 谁选择 Skill | 优点 | 风险 |
| --- | --- | --- | --- |
| 显式触发 | 用户或上层工作流指定 `$skill-name` | 确定、易复现 | 用户需要知道 Skill 名称 |
| 隐式触发 | 模型根据 `description` 选择 | 自然、低操作成本 | 描述过宽会误触发，过窄会漏触发 |

触发信息必须写在 Frontmatter `description`。把“什么时候使用”只写在正文无效，因为正文要等触发后才加载。

高风险 Skill 可以在 `agents/openai.yaml` 中关闭隐式触发，保留显式调用。[1]


### 7.7 安装范围、分发与禁用

Codex 当前支持以下发现范围：[1]

| 范围 | 位置 | 用途 |
| --- | --- | --- |
| Repo | 从 `$CWD/.agents/skills` 向仓库根逐级扫描 | 团队或子目录专用流程 |
| User | `$HOME/.agents/skills` | 个人跨项目能力 |
| Admin | `/etc/codex/skills` | 机器或容器统一能力 |
| System | Codex 内置 | 通用官方能力 |

本地目录适合开发和仓库协作；需要跨团队安装、组合多个 Skill 或携带连接器时使用 Plugin 分发。禁用 Skill 可以通过 Codex 配置完成，更新未被发现时需要重新扫描或重启客户端。[1]


## 8 Skill 实验：需求文档质量审查

### 8.1 实验目标

用户请求：审查一份需求文档，找出验收标准、权限、异常处理和非功能需求中的缺口。

本实验安装三个 Skill：需求审查、会议总结、CSV 分析。模型先根据三份元数据选择 Skill，再读取被选中的 `SKILL.md`，决定使用哪些资源，执行脚本，读取 Observation，最后生成审查报告。

CLI 案例不会进入这条链路。Skill Script 只负责确定性提取 Markdown 结构，模型负责结合领域规则做语义判断。


### 8.2 准备 Skill 工作区和模型配置

代码在系统临时目录构造 Skill，不会安装到真实的 `~/.agents/skills`。环境变量沿用全课程配置：

```dotenv
LLM_API_KEY=你的密钥
LLM_BASE_URL=https://api.deepseek.com
LLM_MODEL=你账号可用的模型名
RUN_LIVE_LLM=1
```

也可以使用 `DEEPSEEK_API_KEY`。Notebook 不打印 Key；仓库验证时强制 `RUN_LIVE_LLM=0`。


In [13]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

from dotenv import find_dotenv, load_dotenv


ENV_PATH = find_dotenv(usecwd=True)
load_dotenv(ENV_PATH, override=False)

LLM_API_KEY = (
    os.getenv("LLM_API_KEY")
    or os.getenv("DEEPSEEK_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)
LLM_BASE_URL = os.getenv("LLM_BASE_URL") or "https://api.deepseek.com"
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_STRUCTURED_MODE = os.getenv("LLM_STRUCTURED_MODE", "json_object")
RUN_LIVE_LLM = os.getenv("RUN_LIVE_LLM", "0") == "1"
RUN_LIVE_EVAL = os.getenv("RUN_LIVE_EVAL", "0") == "1"

SKILL_WORKSPACE = Path(tempfile.gettempdir()) / "agent-course-ch09-skills"
if SKILL_WORKSPACE.exists():
    shutil.rmtree(SKILL_WORKSPACE)

SKILLS_HOME = SKILL_WORKSPACE / ".agents" / "skills"
INPUTS_DIR = SKILL_WORKSPACE / "inputs"
SKILLS_HOME.mkdir(parents=True)
INPUTS_DIR.mkdir(parents=True)

print({
    "env_file": bool(ENV_PATH),
    "provider_configured": bool(LLM_API_KEY and LLM_MODEL),
    "base_url": LLM_BASE_URL,
    "model": LLM_MODEL or "<not configured>",
    "run_live_llm": RUN_LIVE_LLM,
    "run_live_eval": RUN_LIVE_EVAL,
})


{'env_file': True, 'provider_configured': True, 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-v4-pro', 'run_live_llm': False, 'run_live_eval': False}


### 8.3 编写 Skill、Script、Reference、Asset 和 Evals

`SKILL.md` 只保存主流程。评审规则放入 reference，Markdown 结构提取放入 script，报告结构放入 asset，评测提示词放入 evals。


In [14]:
REVIEW_SKILL = """---
name: review-requirements
description: 审查产品需求、PRD、用户故事和验收标准，识别目标、权限、异常流程、可测试性和非功能需求缺口。用户要求评审需求文档、检查 PRD、补充验收条件或分析需求风险时使用。
---

# Review requirements

1. 确认输入文档路径存在；缺少文件时停止并请求路径。
2. 运行 `scripts/extract_sections.py --input <path>`，读取结构化章节清单。
3. 读取 `references/review-rules.md`，只按其中规则评审。
4. 使用文档原文和脚本 Observation 生成发现；每条发现必须包含规则 ID、原文证据和修改建议。
5. 使用 `assets/review-report-template.md` 的字段组织结果。
6. 不修改原需求文档；证据不足时输出待确认问题。

## Failure handling

- 脚本退出码非 0：返回 stderr，不继续生成结论。
- 文档没有可识别标题：说明格式问题，并基于原文做有限审查。
- 发现高风险权限或数据问题：标记 P0，等待产品和安全负责人确认。
"""

REVIEW_RULES = """# Requirement review rules

- R1 Goal: 目标、用户价值和范围边界明确。
- R2 Actor and permission: 角色、数据范围、授权和越权行为明确。
- R3 Acceptance: 验收条件可观察、可度量、可测试。
- R4 Failure path: 超时、重复提交、部分失败、空数据和恢复方式明确。
- R5 Non-functional: 性能、容量、安全、审计、兼容和可用性要求明确。

严重度：阻塞发布或产生安全风险为 P0；影响主要流程为 P1；表达和补充信息问题为 P2。
"""

REPORT_TEMPLATE = """# 需求审查报告

## 结论
{{summary}}

## 发现
{{findings}}

## 缺失章节
{{missing_sections}}

## 待确认问题
{{questions}}
"""

EXTRACT_SCRIPT = r"""#!/usr/bin/env python3
import argparse
import json
import re
import sys
from pathlib import Path


def main() -> int:
    parser = argparse.ArgumentParser(description="提取 Markdown 需求文档的标题和结构信号")
    parser.add_argument("--input", required=True, help="Markdown 文件路径")
    args = parser.parse_args()

    path = Path(args.input)
    if not path.is_file():
        print(json.dumps({"code": "FILE_NOT_FOUND", "path": str(path)}, ensure_ascii=False), file=sys.stderr)
        return 2

    text = path.read_text(encoding="utf-8")
    headings = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)
        if match:
            headings.append({"level": len(match.group(1)), "title": match.group(2), "line": line_no})

    expected = {
        "goal": ["目标", "价值", "范围"],
        "actor_permission": ["角色", "权限", "授权"],
        "acceptance": ["验收", "测试"],
        "failure_path": ["异常", "失败", "错误", "恢复"],
        "non_functional": ["非功能", "性能", "安全", "容量", "可用性"],
    }
    titles = " ".join(item["title"] for item in headings)
    signals = {
        key: any(keyword in titles for keyword in keywords)
        for key, keywords in expected.items()
    }
    result = {
        "file": str(path),
        "heading_count": len(headings),
        "headings": headings,
        "section_signals": signals,
        "line_count": len(text.splitlines()),
    }
    print(json.dumps(result, ensure_ascii=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
"""

OPENAI_YAML = """interface:
  display_name: "需求文档审查"
  short_description: "检查需求完整性、可测试性和风险"
  default_prompt: "使用需求审查 Skill 评审指定文档"
policy:
  allow_implicit_invocation: true
"""

EVALS = {
    "skill_name": "review-requirements",
    "evals": [
        {
            "id": "positive-prd",
            "prompt": "请审查 inputs/export-orders.md，指出需求缺口",
            "expected_output": "选择 review-requirements，并按规则输出带证据的发现",
            "files": ["inputs/export-orders.md"],
            "assertions": ["使用审查规则", "发现权限与异常流程缺口", "每条发现包含证据"],
        },
        {
            "id": "negative-meeting",
            "prompt": "总结今天的会议记录",
            "expected_output": "不触发 review-requirements",
            "files": [],
            "assertions": ["路由到会议总结 Skill 或不选择 Skill"],
        },
        {
            "id": "missing-file",
            "prompt": "审查 inputs/missing.md",
            "expected_output": "报告文件不存在，不生成需求结论",
            "files": [],
            "assertions": ["停止执行", "返回 FILE_NOT_FOUND"],
        },
    ],
}

REQUIREMENT_TEXT = """# 批量导出订单

## 背景
运营同学需要按筛选条件导出订单，当前只能逐页复制。

## 用户故事
作为运营人员，我希望点击导出后获得 CSV 文件，以便离线分析。

## 验收标准
1. 点击导出后生成 CSV。
2. 文件包含订单号、金额和创建时间。
3. 十万条订单应当可以导出。
"""


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


review_dir = SKILLS_HOME / "review-requirements"
write_text(review_dir / "SKILL.md", REVIEW_SKILL)
write_text(review_dir / "agents" / "openai.yaml", OPENAI_YAML)
write_text(review_dir / "scripts" / "extract_sections.py", EXTRACT_SCRIPT)
write_text(review_dir / "references" / "review-rules.md", REVIEW_RULES)
write_text(review_dir / "assets" / "review-report-template.md", REPORT_TEMPLATE)
write_text(review_dir / "evals" / "evals.json", json.dumps(EVALS, ensure_ascii=False, indent=2))

# 两个干扰 Skill 让模型真正执行路由选择，而不是默认只有一个答案。
write_text(
    SKILLS_HOME / "summarize-meeting" / "SKILL.md",
    """---
name: summarize-meeting
description: 总结会议记录，提取决策、待办和负责人。用户要求整理会议纪要、提炼会议结论或跟踪会后任务时使用。
---
读取会议记录，输出决策和待办。
""",
)
write_text(
    SKILLS_HOME / "analyze-csv" / "SKILL.md",
    """---
name: analyze-csv
description: 分析 CSV 数据，执行清洗、统计和表格检查。用户要求处理 CSV 文件、计算指标或检查数据质量时使用。
---
读取 CSV，确认字段后执行分析。
""",
)

REQUIREMENT_PATH = INPUTS_DIR / "export-orders.md"
write_text(REQUIREMENT_PATH, REQUIREMENT_TEXT)

print("skill workspace:", SKILL_WORKSPACE)


skill workspace: /var/folders/7h/j5fl68l96lqf9f6mp9qbdklm0000gn/T/agent-course-ch09-skills


### 8.4 观察目录，不急着调用模型

先确认 Agent 启动时能发现哪些文件。这里还没有加载 reference，也没有执行 script。


In [15]:
def print_tree(root: Path) -> None:
    for path in sorted(root.rglob("*")):
        if path.is_file():
            print(path.relative_to(root))


print_tree(SKILLS_HOME)


analyze-csv/SKILL.md
review-requirements/SKILL.md
review-requirements/agents/openai.yaml
review-requirements/assets/review-report-template.md
review-requirements/evals/evals.json
review-requirements/references/review-rules.md
review-requirements/scripts/extract_sections.py
summarize-meeting/SKILL.md


### 8.5 静态校验与脚本测试

正式工具可以使用 `skills-ref validate <skill-dir>` 或客户端提供的校验器。[2] 下面实现本章可观察的最小校验：Frontmatter、目录名、描述、文件引用和 Python 语法。


In [16]:
def parse_frontmatter(text: str) -> tuple[dict[str, str], str]:
    if not text.startswith("---\n"):
        raise ValueError("SKILL.md 必须以 YAML Frontmatter 开始")
    _, raw_header, body = text.split("---\n", 2)
    metadata: dict[str, str] = {}
    for line in raw_header.strip().splitlines():
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip()
    return metadata, body.strip()


def validate_skill_dir(skill_dir: Path) -> dict[str, Any]:
    skill_file = skill_dir / "SKILL.md"
    metadata, body = parse_frontmatter(skill_file.read_text(encoding="utf-8"))
    name = metadata.get("name", "")
    description = metadata.get("description", "")

    assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", name), "name 格式非法"
    assert name == skill_dir.name, "name 必须与目录名一致"
    assert 1 <= len(name) <= 64, "name 长度非法"
    assert 1 <= len(description) <= 1024, "description 长度非法"
    assert body, "Skill 正文不能为空"

    # 检查正文提到的一层资源路径，避免运行时才发现文件不存在。
    referenced_paths = set(re.findall(r"(?:scripts|references|assets)/[A-Za-z0-9_.-]+", body))
    missing = sorted(path for path in referenced_paths if not (skill_dir / path).is_file())
    assert not missing, f"缺少资源：{missing}"

    for script in (skill_dir / "scripts").glob("*.py") if (skill_dir / "scripts").exists() else []:
        result = subprocess.run(
            [sys.executable, "-m", "py_compile", str(script)],
            capture_output=True,
            text=True,
            timeout=10,
        )
        assert result.returncode == 0, result.stderr

    return {"name": name, "description": description, "resources": sorted(referenced_paths)}


validation_result = validate_skill_dir(review_dir)
print(json.dumps(validation_result, ensure_ascii=False, indent=2))


{
  "name": "review-requirements",
  "description": "审查产品需求、PRD、用户故事和验收标准，识别目标、权限、异常流程、可测试性和非功能需求缺口。用户要求评审需求文档、检查 PRD、补充验收条件或分析需求风险时使用。",
  "resources": [
    "assets/review-report-template.md",
    "references/review-rules.md",
    "scripts/extract_sections.py"
  ]
}


### 8.6 安装与发现：只读取元数据

Host 扫描 Skill Scope，提取名称、描述和路径。此时不会把全部 `SKILL.md` 和资源塞入模型上下文。


In [17]:
def discover_skills(skills_home: Path) -> list[dict[str, str]]:
    catalog = []
    for skill_file in sorted(skills_home.glob("*/SKILL.md")):
        metadata, _ = parse_frontmatter(skill_file.read_text(encoding="utf-8"))
        catalog.append({
            "name": metadata["name"],
            "description": metadata["description"],
            "path": str(skill_file.parent),
        })
    return catalog


skill_catalog = discover_skills(SKILLS_HOME)
print(json.dumps(skill_catalog, ensure_ascii=False, indent=2))

assert [item["name"] for item in skill_catalog] == [
    "analyze-csv",
    "review-requirements",
    "summarize-meeting",
]


[
  {
    "name": "analyze-csv",
    "description": "分析 CSV 数据，执行清洗、统计和表格检查。用户要求处理 CSV 文件、计算指标或检查数据质量时使用。",
    "path": "/var/folders/7h/j5fl68l96lqf9f6mp9qbdklm0000gn/T/agent-course-ch09-skills/.agents/skills/analyze-csv"
  },
  {
    "name": "review-requirements",
    "description": "审查产品需求、PRD、用户故事和验收标准，识别目标、权限、异常流程、可测试性和非功能需求缺口。用户要求评审需求文档、检查 PRD、补充验收条件或分析需求风险时使用。",
    "path": "/var/folders/7h/j5fl68l96lqf9f6mp9qbdklm0000gn/T/agent-course-ch09-skills/.agents/skills/review-requirements"
  },
  {
    "name": "summarize-meeting",
    "description": "总结会议记录，提取决策、待办和负责人。用户要求整理会议纪要、提炼会议结论或跟踪会后任务时使用。",
    "path": "/var/folders/7h/j5fl68l96lqf9f6mp9qbdklm0000gn/T/agent-course-ch09-skills/.agents/skills/summarize-meeting"
  }
]


### 8.7 渐进披露的实际加载量

字符数不等于 Token 数，但可以直接观察三层材料的数量级。启动阶段只承担所有 Skill 元数据；触发后才加载一份正文；资源仍保持未加载。


In [18]:
catalog_text = json.dumps(skill_catalog, ensure_ascii=False)
selected_metadata, selected_body = parse_frontmatter((review_dir / "SKILL.md").read_text(encoding="utf-8"))
resource_texts = [
    (review_dir / "references" / "review-rules.md").read_text(encoding="utf-8"),
    (review_dir / "assets" / "review-report-template.md").read_text(encoding="utf-8"),
    (review_dir / "scripts" / "extract_sections.py").read_text(encoding="utf-8"),
]

loading_cost = {
    "startup_catalog_chars": len(catalog_text),
    "selected_skill_body_chars": len(selected_body),
    "all_selected_resources_chars": sum(len(text) for text in resource_texts),
}
print(json.dumps(loading_cost, ensure_ascii=False, indent=2))

assert loading_cost["startup_catalog_chars"] < sum(loading_cost.values())


{
  "startup_catalog_chars": 677,
  "selected_skill_body_chars": 406,
  "all_selected_resources_chars": 1919
}


### 8.8 Script 独立执行：确定性事实先于模型判断

模型稍后会选择是否运行脚本。这里先单独验证脚本接口、退出码和 JSON 输出，避免把脚本错误误判成模型错误。


In [19]:
@dataclass
class ScriptResult:
    exit_code: int
    stdout: str
    stderr: str

    def json(self) -> dict[str, Any]:
        return json.loads(self.stdout) if self.stdout else {}


def safe_skill_path(skill_dir: Path, relative_path: str, expected_prefix: str) -> Path:
    if not relative_path.startswith(expected_prefix + "/"):
        raise ValueError(f"资源必须位于 {expected_prefix}/")
    path = (skill_dir / relative_path).resolve()
    if not path.is_relative_to(skill_dir.resolve()) or not path.is_file():
        raise ValueError(f"非法或不存在的 Skill 资源：{relative_path}")
    return path


def run_skill_script(skill_dir: Path, relative_path: str, input_path: Path) -> ScriptResult:
    script_path = safe_skill_path(skill_dir, relative_path, "scripts")
    completed = subprocess.run(
        [sys.executable, str(script_path), "--input", str(input_path.resolve())],
        cwd=skill_dir,
        capture_output=True,
        text=True,
        timeout=10,
    )
    return ScriptResult(completed.returncode, completed.stdout.strip(), completed.stderr.strip())


script_test_result = run_skill_script(review_dir, "scripts/extract_sections.py", REQUIREMENT_PATH)
print("exit_code=", script_test_result.exit_code)
print(script_test_result.stdout)

assert script_test_result.exit_code == 0
assert script_test_result.json()["section_signals"]["acceptance"] is True
assert script_test_result.json()["section_signals"]["actor_permission"] is False


exit_code= 0
{"file": "/private/var/folders/7h/j5fl68l96lqf9f6mp9qbdklm0000gn/T/agent-course-ch09-skills/inputs/export-orders.md", "heading_count": 4, "headings": [{"level": 1, "title": "批量导出订单", "line": 1}, {"level": 2, "title": "背景", "line": 3}, {"level": 2, "title": "用户故事", "line": 6}, {"level": 2, "title": "验收标准", "line": 9}], "section_signals": {"goal": false, "actor_permission": false, "acceptance": true, "failure_path": false, "non_functional": false}, "line_count": 12}


### 8.9 接入真实模型

接下来不再使用关键词函数或固定路由。模型先看到三份 Skill 元数据，输出选择结果；Host 再加载被选中的正文。`call_json_model` 使用 OpenAI-compatible Chat Completions，适配当前 DeepSeek 配置。


In [20]:
from openai import OpenAI


def parse_json_object(text: str) -> dict[str, Any]:
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("模型响应中没有 JSON 对象")
    return json.loads(text[start:end + 1])


def create_model_client() -> OpenAI:
    if not LLM_API_KEY or not LLM_MODEL:
        raise RuntimeError("需要配置 LLM_API_KEY/DEEPSEEK_API_KEY 和 LLM_MODEL")
    options: dict[str, Any] = {"api_key": LLM_API_KEY}
    if LLM_BASE_URL:
        options["base_url"] = LLM_BASE_URL
    return OpenAI(**options)


def call_json_model(client: OpenAI, system: str, user: str) -> dict[str, Any]:
    request: dict[str, Any] = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0,
    }
    if LLM_STRUCTURED_MODE == "json_object":
        request["response_format"] = {"type": "json_object"}
    response = client.chat.completions.create(**request)
    return parse_json_object(response.choices[0].message.content or "")


### 8.10 生命周期第一段：模型执行隐式触发

路由请求只包含用户目标和 Skill 元数据。模型看不到任何 Skill 正文、脚本源码或评审规则。


In [21]:
def route_skill_with_model(
    client: OpenAI,
    user_request: str,
    catalog: list[dict[str, str]],
) -> dict[str, Any]:
    public_catalog = [
        {"name": item["name"], "description": item["description"]}
        for item in catalog
    ]
    return call_json_model(
        client,
        system=(
            "你是 Skill 路由器。只根据用户请求和 Skill 的 name/description 选择。"
            "只有明确匹配时才选择；无匹配返回 null。"
            '只返回 JSON：{"skill_name": string|null, "reason": string}。'
        ),
        user=json.dumps({"request": user_request, "skills": public_catalog}, ensure_ascii=False),
    )


### 8.11 生命周期第二段：加载正文，由模型选择资源

触发后，Host 加载选中 Skill 的完整正文，并向模型提供可用资源路径。模型决定读取 reference、运行 script 和使用 asset；Host 校验所有相对路径。


In [22]:
def list_skill_resources(skill_dir: Path) -> list[str]:
    return [
        str(path.relative_to(skill_dir))
        for folder in ("scripts", "references", "assets")
        for path in sorted((skill_dir / folder).glob("*"))
        if path.is_file()
    ]


def choose_resources_with_model(
    client: OpenAI,
    user_request: str,
    skill_body: str,
    resources: list[str],
) -> dict[str, Any]:
    return call_json_model(
        client,
        system=(
            "你正在执行已经触发的 Skill。根据 SKILL.md 决定本任务需要的资源。"
            "只能选择给出的相对路径，不得生成命令参数或其他路径。"
            '只返回 JSON：{"references": [string], "scripts": [string], '
            '"assets": [string], "reason": string}。'
        ),
        user=json.dumps({
            "request": user_request,
            "skill_body": skill_body,
            "available_resources": resources,
        }, ensure_ascii=False),
    )


def validate_resource_plan(skill_dir: Path, plan: dict[str, Any]) -> dict[str, list[str]]:
    validated: dict[str, list[str]] = {}
    for key, prefix in (("references", "references"), ("scripts", "scripts"), ("assets", "assets")):
        values = plan.get(key, [])
        if not isinstance(values, list):
            raise ValueError(f"{key} 必须是数组")
        validated[key] = []
        for relative_path in values:
            safe_skill_path(skill_dir, relative_path, prefix)
            validated[key].append(relative_path)
    return validated


### 8.12 生命周期第三段：执行资源、回传 Observation、生成结果

Host 执行模型选择的脚本，把 stdout 解析为 Observation；reference 和 asset 也只在被选择后读取。最终模型同时看到文档原文、领域规则、脚本事实和输出模板。


In [23]:
def complete_review_with_model(
    client: OpenAI,
    user_request: str,
    requirement_text: str,
    reference_texts: dict[str, str],
    script_observations: dict[str, Any],
    asset_texts: dict[str, str],
) -> dict[str, Any]:
    return call_json_model(
        client,
        system=(
            "你是需求审查 Agent。严格依据给出的评审规则和文档证据。"
            "每条 finding 必须包含 severity、rule_id、evidence、recommendation。"
            "不得虚构文档内容。"
            '只返回 JSON：{"summary": string, "findings": [object], '
            '"missing_sections": [string], "questions": [string]}。'
        ),
        user=json.dumps({
            "request": user_request,
            "requirement": requirement_text,
            "references": reference_texts,
            "script_observations": script_observations,
            "output_assets": asset_texts,
        }, ensure_ascii=False),
    )


def run_live_skill_lifecycle(user_request: str, input_path: Path) -> dict[str, Any]:
    client = create_model_client()

    # 1. Discovery / Trigger：模型只看元数据。
    routing = route_skill_with_model(client, user_request, skill_catalog)
    selected_name = routing.get("skill_name")
    if not selected_name:
        return {"routing": routing, "status": "no_skill_selected"}

    selected = next((item for item in skill_catalog if item["name"] == selected_name), None)
    if selected is None:
        raise ValueError(f"模型选择了未安装的 Skill：{selected_name}")

    # 2. Load：选中后才读取完整 SKILL.md。
    skill_dir = Path(selected["path"])
    _, skill_body = parse_frontmatter((skill_dir / "SKILL.md").read_text(encoding="utf-8"))
    available_resources = list_skill_resources(skill_dir)

    # 3. Resource planning：模型选择本次真正需要的资源。
    raw_plan = choose_resources_with_model(client, user_request, skill_body, available_resources)
    resource_plan = validate_resource_plan(skill_dir, raw_plan)

    # 4. Execute：Host 校验路径后读取文件、执行脚本。
    references = {
        path: safe_skill_path(skill_dir, path, "references").read_text(encoding="utf-8")
        for path in resource_plan["references"]
    }
    assets = {
        path: safe_skill_path(skill_dir, path, "assets").read_text(encoding="utf-8")
        for path in resource_plan["assets"]
    }
    observations: dict[str, Any] = {}
    for script_path in resource_plan["scripts"]:
        result = run_skill_script(skill_dir, script_path, input_path)
        if result.exit_code != 0:
            return {
                "routing": routing,
                "resource_plan": raw_plan,
                "status": "script_failed",
                "script_error": {"exit_code": result.exit_code, "stderr": result.stderr},
            }
        observations[script_path] = result.json()

    # 5. Observe / Complete：模型读取执行事实后生成最终报告。
    report = complete_review_with_model(
        client,
        user_request,
        input_path.read_text(encoding="utf-8"),
        references,
        observations,
        assets,
    )
    return {
        "routing": routing,
        "loaded_skill": selected_name,
        "resource_plan": raw_plan,
        "script_observations": observations,
        "report": report,
        "status": "completed",
    }


### 8.13 运行完整真实生命周期

设置 `RUN_LIVE_LLM=1` 后会产生三次模型调用：路由、资源选择、最终审查。代码断言模型选择了需求审查 Skill、执行了结构提取脚本，并输出带证据的发现。

仓库最终保存的 Notebook 会以 `RUN_LIVE_LLM=0` 执行，不保存真实 API 响应。


In [24]:
USER_REQUEST = f"请审查需求文档 {REQUIREMENT_PATH}，指出上线前必须补充的内容"
live_skill_trace: dict[str, Any] | None = None

if RUN_LIVE_LLM:
    live_skill_trace = run_live_skill_lifecycle(USER_REQUEST, REQUIREMENT_PATH)
    print(json.dumps(live_skill_trace, ensure_ascii=False, indent=2))

    assert live_skill_trace["status"] == "completed"
    assert live_skill_trace["loaded_skill"] == "review-requirements"
    assert "scripts/extract_sections.py" in live_skill_trace["resource_plan"]["scripts"]
    assert "references/review-rules.md" in live_skill_trace["resource_plan"]["references"]
    assert live_skill_trace["report"]["findings"]
    assert all(
        {"severity", "rule_id", "evidence", "recommendation"} <= finding.keys()
        for finding in live_skill_trace["report"]["findings"]
    )
else:
    print("live skill lifecycle skipped: set RUN_LIVE_LLM=1 and configure API credentials")


live skill lifecycle skipped: set RUN_LIVE_LLM=1 and configure API credentials


### 8.14 这次实验证明了什么

1. 模型在正文加载前只根据元数据选择 Skill。
2. Host 只加载被选中的 `SKILL.md`。
3. 模型根据正文选择 reference、script 和 asset。
4. Script 产生外部 Observation，模型读取后再生成结论。
5. 路径、参数、执行权限和错误处理仍由 Host 控制。

旧版 6.4 直接调用 Python 工作流，只证明了流程函数能够运行。当前实验让模型真实参与 Skill 的关键决策点，同时保留确定性执行边界。


## 9 Skill 的评测、迭代与生产边界

### 9.1 Skill 需要四组评测

| 评测 | 输入 | 主要指标 |
| --- | --- | --- |
| 触发评测 | 正例、负例、相邻 Skill、显式调用 | 召回率、误触发率 |
| 过程评测 | Transcript、资源计划、工具调用 | 是否加载正确资源、是否越权、失败恢复 |
| 结果评测 | 最终产物和断言 | 任务通过率、证据质量、格式正确率 |
| 成本评测 | Token、模型调用、脚本耗时 | 增量质量是否覆盖增量成本 |

Agent Skills 官方评测建议对同一测试分别运行 with-skill 与 without-skill，保存输出、断言、耗时和 Token，再进行自动评分和人工复查。[4]


In [25]:
eval_payload = json.loads((review_dir / "evals" / "evals.json").read_text(encoding="utf-8"))

assert eval_payload["skill_name"] == "review-requirements"
assert len(eval_payload["evals"]) == 3
assert {item["id"] for item in eval_payload["evals"]} == {
    "positive-prd",
    "negative-meeting",
    "missing-file",
}
assert all(item["assertions"] for item in eval_payload["evals"])

print(json.dumps({
    "eval_count": len(eval_payload["evals"]),
    "cases": [item["id"] for item in eval_payload["evals"]],
    "live_trigger_eval_enabled": RUN_LIVE_EVAL,
}, ensure_ascii=False, indent=2))


{
  "eval_count": 3,
  "cases": [
    "positive-prd",
    "negative-meeting",
    "missing-file"
  ],
  "live_trigger_eval_enabled": false
}


### 9.2 使用真实模型评测触发边界

`RUN_LIVE_EVAL=1` 时，下面会让模型分别处理需求审查、会议总结和缺失需求文件三条请求。缺失文件仍应触发需求审查 Skill，随后由执行阶段返回 `FILE_NOT_FOUND`；会议总结不能误触发需求审查。


In [26]:
TRIGGER_EXPECTATIONS = {
    "positive-prd": "review-requirements",
    "negative-meeting": "summarize-meeting",
    "missing-file": "review-requirements",
}

if RUN_LIVE_EVAL:
    eval_client = create_model_client()
    trigger_results = []
    for case in eval_payload["evals"]:
        routing = route_skill_with_model(eval_client, case["prompt"], skill_catalog)
        actual = routing.get("skill_name")
        expected = TRIGGER_EXPECTATIONS[case["id"]]
        trigger_results.append({
            "id": case["id"],
            "expected": expected,
            "actual": actual,
            "passed": actual == expected,
            "reason": routing.get("reason", ""),
        })

    print(json.dumps(trigger_results, ensure_ascii=False, indent=2))
    assert all(item["passed"] for item in trigger_results)
else:
    print("live trigger eval skipped: set RUN_LIVE_EVAL=1 and configure API credentials")


live trigger eval skipped: set RUN_LIVE_EVAL=1 and configure API credentials


### 9.3 评测后的迭代顺序

1. **误触发**：先收紧或补充 `description`，不要在正文修，因为正文尚未加载。
2. **漏步骤**：修改 `SKILL.md` 主流程和失败处理。
3. **重复生成代码**：提取为 Script，增加输入输出断言。
4. **上下文过大**：把详细规则迁移到 reference，按条件读取。
5. **输出不一致**：增加 asset/template 和可机器验证的 Schema。
6. **新版本回归**：与旧 Skill 快照做盲测，不只与无 Skill 基线比较。

版本信息可以记录在组织自己的 metadata、Plugin 版本或发布清单中；客户端对自定义版本字段的解释可能不同。升级时同时验证触发、资源路径、脚本接口和输出契约。


### 9.4 安全、权限与供应链

Skill 文件和脚本属于可执行供应链内容。安装前需要审查来源、指令、脚本、网络访问和依赖版本。

| 风险 | 控制方式 |
| --- | --- |
| 恶意指令诱导读取秘密 | 沙箱、秘密隔离、工具权限和人工审批 |
| 模型选择任意路径 | 相对路径白名单、根目录校验、拒绝路径穿越 |
| Script 下载浮动依赖 | 锁版本、校验来源、离线构建或制品签名 |
| Script 交互等待 | 非交互参数、超时和明确退出码 |
| 隐式触发高风险操作 | 关闭隐式调用，要求显式触发和人审 |
| Skill 更新改变行为 | 版本固定、变更审查和回归评测 |

渐进披露只控制上下文加载量，不提供安全隔离。`allowed-tools` 和产品级依赖声明也需要运行时策略配合。[1][2]


## 10 Skill 与 CLI 如何搭配

两项技术都学完后，组合关系很简单：Skill 正文可以把 Agent-facing CLI 列为一个可用动作，并说明调用顺序和错误恢复；CLI 不需要知道调用者是否使用了 Skill。

```mermaid
sequenceDiagram
    participant U as User
    participant M as Model
    participant S as Skill
    participant C as CLI
    participant A as Business API
    U->>M: 完成一个多步骤业务目标
    M->>S: 加载流程和失败处理
    S-->>M: 先查询、再预演、确认后写入
    M->>C: 执行结构化命令
    C->>A: 调用业务 API
    A-->>C: 结果
    C-->>M: exit code + JSON Observation
```

组合前提：CLI 已经独立稳定，Skill 只引用公开命令契约。不要把业务权限、幂等和审计写进 Skill 文本后从 CLI 删除。


### 10.1 选型表

| 需求 | 选择 |
| --- | --- |
| 单次任务约束 | Prompt |
| 可复用的模型工作流程 | Skill |
| Skill 内部重复且确定的处理 | Skill Script |
| 稳定的跨调用方业务动作 | Agent-facing CLI |
| 多 Host 的工具发现和远程调用协议 | MCP |
| 强制执行权限、预算、恢复和审批 | Harness / API / CLI |

常见组合：Skill + 文件工具、Skill + MCP、Skill + 浏览器、Skill + Script、Skill + CLI。CLI 也可以独立存在，并被 MCP Server 包装。


## 11 上线检查清单

### 11.1 CLI

- [ ] 输入、输出、退出码、错误码和版本稳定。
- [ ] stdout 与 stderr 分离。
- [ ] 认证、Scope、租户和账户显式绑定。
- [ ] 分页、超时、限流、dry-run、幂等和审计可验证。
- [ ] 凭据不进入参数、日志、Prompt 和 Checkpoint。

### 11.2 Skill

- [ ] Description 覆盖正例、负例和相邻 Skill 边界。
- [ ] `SKILL.md` 保持短小，资源路径存在且只引用一层。
- [ ] Script 非交互、结构化输出、失败码明确并经过测试。
- [ ] Reference、Script、Asset 按需加载，没有重复内容。
- [ ] Host 校验路径、权限、网络、超时和副作用。
- [ ] 触发、过程、结果、成本和旧版本回归均有 Evals。
- [ ] 安装来源、版本、分发和禁用方式明确。


## 12 本章小结

1. Agent-facing CLI 提供稳定执行接口，独立承担参数、认证、授权、输出、错误、幂等和审计。
2. Agent Skill 提供可发现、按需加载、可复用和可评测的程序性知识。
3. Skill 生命周期覆盖编写、校验、安装、发现、触发、加载、执行、Observation、评测、迭代、分发和下线。
4. Script 用于重复且确定的处理；Reference 保存按需领域资料；Asset 服务于输出产物。
5. 模型参与 Skill 触发、资源选择和结果综合，Host 控制资源读取、脚本执行和权限。
6. Skill 与 CLI 可以组合，也可以分别使用。


## 13 与最终项目的关系

第 13 章的产研协作助手会把稳定业务动作整理为 Agent-facing CLI，把需求评审、研发协作和测试分析等多步骤知识整理为 Skill。本章只学习两项技术及组合边界。


## 14 下一步

第 10 章进入 Harness Engineering：谁负责装配模型、Prompt、Skill、工具、CLI、上下文、权限、状态、沙箱、审批、事件流和恢复。


## 参考文献

[1] OPENAI. Build skills[EB/OL]. [2026-07-17]. https://learn.chatgpt.com/docs/build-skills.

[2] AGENT SKILLS. Specification[EB/OL]. [2026-07-17]. https://agentskills.io/specification.

[3] AGENT SKILLS. Using scripts in skills[EB/OL]. [2026-07-17]. https://agentskills.io/skill-creation/using-scripts.

[4] AGENT SKILLS. Evaluating skill output quality[EB/OL]. [2026-07-17]. https://agentskills.io/skill-creation/evaluating-skills.

[5] GOOGLE WORKSPACE. Google Workspace CLI[EB/OL]. [2026-07-17]. https://github.com/googleworkspace/cli.

[6] GOOGLE. OAuth 2.0 policies and best practices[EB/OL]. [2026-07-17]. https://developers.google.com/identity/protocols/oauth2/resources/best-practices.

[7] GOOGLE. Building a client library for the Discovery API[EB/OL]. [2026-07-17]. https://developers.google.com/discovery/v1/building-a-client-library.
